In [ ]:
from pathlib import Path
from frameit.core.settings_class import SimulationConfig
from frameit.core.runner import FrameitRunner

import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import xarray as xr
import math

# conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/arome_oper_batsirai.yml"))
# conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/arome_oper_belna.yml"))

conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/mnh_chido.yml"))
# conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/mnh_ianos.yml"))

conf.printID()

runner = FrameitRunner(conf)
runner.run()

In [ ]:
runner.dict_collocated_crop_user

Les codes suivant test les erreurs du à la projection dans un repère plan lorsque l'on utilise une interpolation entre deux grille pour la projection polaire
Ces erreurs grandissent avec la taille de box, ce qui est normale étant donnée que le plan est une approximation de la sphère
Ces erreurs devenant trop grande => on préfère utiliser xESMF plutot que l'interpolation classique

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from pyproj import Geod

from frameit.processing.polar.polar_grid import PolarLonLatGrid

def check_polar_grid_consistency(
    ds_surface: xr.Dataset,
    conf,
    *,
    t: int = 0,
    make_plots: bool = True,
):
    """
    Verify that the polar lon/lat grid is consistent with the cartesian box lon/lat.

    Checks
    ------
    1) r=0 point of the polar grid equals the box center lon/lat (x_box=0, y_box=0).
    2) Geodesic forward mapping applied to the cartesian metric grid (x_box_km, y_box_km)
       reproduces the dataset lon/lat field (distance error map in meters).

    Notes
    -----
    - If ds_surface lon/lat are produced by a map projection (not geodesics),
      small residuals (order 10–100 m over O(100 km)) can be normal.
    """
    # Build polar grid (lon/lat and x_km/y_km)
    grid = PolarLonLatGrid.from_conf(conf)
    ds_polar = grid.build(ds_surface)

    time_dim = grid.time_dim
    lon_name = grid.lon_name
    lat_name = grid.lat_name

    geod = Geod(ellps="WGS84")

    # ------------------------------------------------------------
    # 1) Center consistency: polar(r=0) must equal ds center
    # ------------------------------------------------------------
    lon0 = ds_surface[lon_name].sel(x_box=0, y_box=0)
    lat0 = ds_surface[lat_name].sel(x_box=0, y_box=0)

    if time_dim in lon0.dims:
        lon0_t = float(lon0.isel({time_dim: t}).values)
        lat0_t = float(lat0.isel({time_dim: t}).values)
    else:
        lon0_t = float(lon0.values)
        lat0_t = float(lat0.values)

    lon_r0 = float(ds_polar["lon"].isel({time_dim: t, "r_km": 0, "theta_deg": 0}).values)
    lat_r0 = float(ds_polar["lat"].isel({time_dim: t, "r_km": 0, "theta_deg": 0}).values)
    _, _, dist_center_m = geod.inv(lon0_t, lat0_t, lon_r0, lat_r0)

    print(f"[Center check] time index t={t}")
    print(f"  ds center lon/lat:    {lon0_t:.6f}, {lat0_t:.6f}")
    print(f"  polar r=0 lon/lat:    {lon_r0:.6f}, {lat_r0:.6f}")
    print(f"  center distance error: {dist_center_m:.3f} m")

    # ------------------------------------------------------------
    # 2) Cartesian-to-geodesic consistency on the full box grid
    # ------------------------------------------------------------
    if "x_box_km" not in ds_surface.coords or "y_box_km" not in ds_surface.coords:
        raise KeyError("ds_surface must provide coords 'x_box_km' and 'y_box_km' (1D).")

    x = np.asarray(ds_surface["x_box_km"].values, dtype=float)  # (x_box,)
    y = np.asarray(ds_surface["y_box_km"].values, dtype=float)  # (y_box,)
    X, Y = np.meshgrid(x, y, indexing="xy")  # (y_box, x_box)

    # Convert metric coordinates to polar (r, azimuth from North, clockwise)
    r_m = 1000.0 * np.hypot(X, Y)
    az_deg = (np.degrees(np.arctan2(X, Y)) + 360.0) % 360.0

    lon_pred, lat_pred, _ = geod.fwd(
        np.full_like(r_m, lon0_t),
        np.full_like(r_m, lat0_t),
        az_deg,
        r_m,
    )

    lon_src = np.asarray(ds_surface[lon_name].isel({time_dim: t}).values, dtype=float)
    lat_src = np.asarray(ds_surface[lat_name].isel({time_dim: t}).values, dtype=float)

    # Great-circle/geodesic distance between dataset lon/lat and geodesic prediction
    _, _, dist_err_m = geod.inv(lon_src, lat_src, lon_pred, lat_pred)
    dist_err_m = np.asarray(dist_err_m, dtype=float)

    print("[Cartesian consistency check]")
    print(f"  max dist error:  {np.nanmax(dist_err_m):.3f} m")
    print(f"  mean dist error: {np.nanmean(dist_err_m):.3f} m")
    print(f"  p95 dist error:  {np.nanpercentile(dist_err_m, 95):.3f} m")

    err_da = xr.DataArray(
        dist_err_m,
        dims=("y_box", "x_box"),
        coords={"y_box": ds_surface["y_box"], "x_box": ds_surface["x_box"]},
        name="dist_err_m",
        attrs={"description": "distance error between ds lon/lat and geodesic mapping from center"},
    )

    if make_plots:
        plt.figure()
        plt.imshow(dist_err_m, origin="lower")
        plt.colorbar(label="distance error (m)")
        plt.title(f"Geodesic vs ds lon/lat, t={t}")
        plt.xlabel("x_box index")
        plt.ylabel("y_box index")
        plt.show()

        plt.figure()
        plt.hist(dist_err_m[np.isfinite(dist_err_m)].ravel(), bins=50)
        plt.title(f"Distance error histogram, t={t}")
        plt.xlabel("distance error (m)")
        plt.ylabel("count")
        plt.show()

    return ds_polar, err_da


# Example usage:
# ds_polar, err = check_polar_grid_consistency(runner.dict_collocated_crop_user["surface"], runner.conf, t=0)

In [ ]:
ds_polar, err = check_polar_grid_consistency(runner.dict_collocated_crop_user["surface"], runner.conf, t=0)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from pyproj import Geod

def check_xy_is_enu(ds_surface, conf, t=0):
    geod = Geod(ellps="WGS84")
    lon_name = conf.name_longitude
    lat_name = conf.name_latitude
    time_dim = conf.name_time_dim

    lon0 = float(ds_surface[lon_name].sel(x_box=0, y_box=0).isel({time_dim: t}).values)
    lat0 = float(ds_surface[lat_name].sel(x_box=0, y_box=0).isel({time_dim: t}).values)

    lon = np.asarray(ds_surface[lon_name].isel({time_dim: t}).values)
    lat = np.asarray(ds_surface[lat_name].isel({time_dim: t}).values)

    # "True" geodesic from center to each grid point
    az_geo, _, r_geo_m = geod.inv(np.full_like(lon, lon0), np.full_like(lat, lat0), lon, lat)

    # "Model" x/y-based polar (assumed ENU)
    x = np.asarray(ds_surface["x_box_km"].values)
    y = np.asarray(ds_surface["y_box_km"].values)
    X, Y = np.meshgrid(x, y, indexing="xy")  # (y_box, x_box)
    r_xy_m = 1000.0 * np.hypot(X, Y)
    az_xy = (np.degrees(np.arctan2(X, Y)) + 360.0) % 360.0

    # Errors
    dr_m = r_geo_m - r_xy_m
    daz = (az_geo - az_xy + 180.0) % 360.0 - 180.0  # wrap to [-180, 180]

    print("Range error (m): max", float(np.nanmax(np.abs(dr_m))), "mean", float(np.nanmean(np.abs(dr_m))))
    print("Azimuth error (deg): max", float(np.nanmax(np.abs(daz))), "mean", float(np.nanmean(np.abs(daz))))

    plt.figure()
    plt.imshow(dr_m, origin="lower")
    plt.colorbar(label="r_geo - r_xy (m)")
    plt.title("Range error map")
    plt.show()

    plt.figure()
    plt.imshow(daz, origin="lower")
    plt.colorbar(label="az_geo - az_xy (deg)")
    plt.title("Azimuth error map")
    plt.show()

    return dr_m, daz


In [ ]:
dr_m, daz = check_xy_is_enu(runner.dict_collocated_crop_user["surface"], conf, t=0)

In [ ]:
import numpy as np

def pct_points_below_threshold(err_m: np.ndarray, dx_m: float, frac: float = 0.1) -> float:
    """
    Percentage of finite points with |err_m| < frac * dx_m.
    """
    thr = frac * dx_m
    m = np.isfinite(err_m)
    if m.sum() == 0:
        return np.nan
    return 100.0 * np.mean(np.abs(err_m[m]) < thr)

# Example (with your dr_m array and dx=2000 m):
pct = pct_points_below_threshold(dr_m, dx_m=2000.0, frac=0.1)
print(f"{pct:.2f}% of points have |dr| < 0.1*dx")